Task 2: End-to-End ML Pipeline with Scikit-learn Pipeline API

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score
)
from sklearn.impute import SimpleImputer
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.dropna(inplace=True)
df.drop(columns=["customerID"], inplace=True)
df["Churn"] = (df["Churn"] == "Yes").astype(int)

target = "Churn"
X = df.drop(columns=[target])
y = df[target]

numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)
lr_proba = lr_pipeline.predict_proba(X_test)[:, 1]

print("=== Logistic Regression Results ===")
print(f"Accuracy: {accuracy_score(y_test, lr_preds):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, lr_proba):.4f}")
print(classification_report(y_test, lr_preds))

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42, n_jobs=-1))
])

rf_param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 10],
    "classifier__min_samples_split": [2, 5]
}

grid_search = GridSearchCV(
    rf_pipeline,
    param_grid=rf_param_grid,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
rf_preds = best_rf.predict(X_test)
rf_proba = best_rf.predict_proba(X_test)[:, 1]

print("\n=== Random Forest Results (Best Params) ===")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Accuracy: {accuracy_score(y_test, rf_preds):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, rf_proba):.4f}")
print(classification_report(y_test, rf_preds))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_lr = confusion_matrix(y_test, lr_preds)
sns.heatmap(cm_lr, annot=True, fmt="d", ax=axes[0], cmap="Blues")
axes[0].set_title("Logistic Regression – Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

cm_rf = confusion_matrix(y_test, rf_preds)
sns.heatmap(cm_rf, annot=True, fmt="d", ax=axes[1], cmap="Greens")
axes[1].set_title("Random Forest – Confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()

rf_classifier = best_rf.named_steps["classifier"]
ohe = best_rf.named_steps["preprocessor"].named_transformers_["cat"].named_steps["encoder"]
cat_feature_names = ohe.get_feature_names_out(categorical_cols).tolist()
all_feature_names = numerical_cols + cat_feature_names

importances = rf_classifier.feature_importances_
feat_df = pd.DataFrame({"feature": all_feature_names, "importance": importances})
feat_df = feat_df.sort_values("importance", ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_df, x="importance", y="feature", palette="viridis")
plt.title("Top 15 Feature Importances – Random Forest")
plt.tight_layout()
plt.savefig("feature_importances.png", dpi=150)
plt.show()

joblib.dump(best_rf, "churn_pipeline_rf.joblib")
joblib.dump(lr_pipeline, "churn_pipeline_lr.joblib")
print("\nPipelines saved: churn_pipeline_rf.joblib, churn_pipeline_lr.joblib")

loaded_pipeline = joblib.load("churn_pipeline_rf.joblib")
sample = X_test.iloc[:5]
sample_preds = loaded_pipeline.predict(sample)
sample_proba = loaded_pipeline.predict_proba(sample)[:, 1]

print("\n=== Sample Predictions from Loaded Pipeline ===")
for i in range(5):
    print(f"Customer {i+1}: Churn={bool(sample_preds[i])}, Probability={sample_proba[i]:.4f}")

=== Logistic Regression Results ===
Accuracy: 0.8038
ROC-AUC:  0.8359
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1033
           1       0.65      0.57      0.61       374

    accuracy                           0.80      1407
   macro avg       0.75      0.73      0.74      1407
weighted avg       0.80      0.80      0.80      1407

Fitting 3 folds for each of 8 candidates, totalling 24 fits

=== Random Forest Results (Best Params) ===
Best Parameters: {'classifier__max_depth': 10, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200}
Accuracy: 0.7918
ROC-AUC:  0.8316
              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1033
           1       0.63      0.51      0.57       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.72      1407
weighted avg       0.78      0.79      0.78      1407

